<a href="https://colab.research.google.com/github/ggalarza1/AI_Finance/blob/main/superintelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import gymnasium as gym
import numpy as np
import pandas as pd
np.random.seed(100)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
env=gym.make('CartPole-v0')

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:511: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


In [6]:
# In Gymnasium, instead of env.seed(), pass the seed parameter to env.reset()
observation, info = env.reset(seed=100)

In [33]:
action_size = env.action_space.n
action_size

np.int64(2)

In [8]:
[env.action_space.sample() for _ in range(10)]

[np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0)]

In [34]:
state_size = env.observation_space.shape[0]
state_size

4

In [10]:
state = env.reset()
state # [cart position, cart velocity, pole angle, pole angular velocity]

(array([0.04736544, 0.00964717, 0.02902632, 0.04103394], dtype=float32), {})

In [12]:
# Gymnasium env.step() returns 5 values: observation, reward, terminated, truncated, info
state, reward, terminated, truncated, info = env.step(env.action_space.sample())
state, reward, terminated, truncated, info

(array([0.0516452 , 0.00880583, 0.02499997, 0.05959459], dtype=float32),
 1.0,
 False,
 False,
 {})

In [38]:
%%time
data_list = []
length = []

for run in range(25000):
  results = []
  done = False
  prev_state, _ = env.reset()
  treward = 0

  while not done:
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)

    results.append({
      's1': prev_state[0],
      's2': prev_state[1],
      's3': prev_state[3],
      'a': action,
      'r': reward,
      's': state,
      't': terminated,
      'tr': truncated
    })

    treward += reward
    prev_state = state
    done = terminated or truncated

  if treward >= 110:
    data_list.append(pd.DataFrame(results))
    length.append(treward)

if data_list:
  data = pd.concat(data_list, ignore_index=True)
else:
  data = pd.DataFrame()

CPU times: user 8.56 s, sys: 176 ms, total: 8.73 s
Wall time: 8.62 s


In [39]:
np.array(length).mean()

np.float64(119.57142857142857)

In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 837 entries, 0 to 836
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   s1      837 non-null    float32
 1   s2      837 non-null    float32
 2   s3      837 non-null    float32
 3   a       837 non-null    int64  
 4   r       837 non-null    float64
 5   s       837 non-null    object 
 6   t       837 non-null    bool   
 7   tr      837 non-null    bool   
dtypes: bool(2), float32(3), float64(1), int64(1), object(1)
memory usage: 31.2+ KB


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [41]:
data.tail()

,s1,s2,s3,a,r,s,t,tr
832,1.076553,0.322112,0.831095,0,1.0,"[1.0829954, 0.1255406, 0.13982576, 1.1598467]",False,False
833,1.082995,0.125541,1.159847,1,1.0,"[1.0855063, 0.31859183, 0.1630227, 0.9140709]",False,False
834,1.085506,0.318592,0.914071,1,1.0,"[1.091878, 0.5111781, 0.18130411, 0.6767384]",False,False
835,1.091878,0.511178,0.676738,0,1.0,"[1.1021017, 0.3140621, 0.19483888, 1.020578]",False,False
836,1.102102,0.314062,1.020578,0,1.0,"[1.1083829, 0.116953395, 0.21525045, 1.3675677]",True,False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [42]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [43]:
import tensorflow as tf
tf.random.set_seed(100)

In [44]:
from keras.layers import Dense
from keras.models import Sequential
from keras.layers import Input
#Using TensorFlow backend.

In [45]:
model = Sequential()
model.add(Input(shape=(env.observation_space.shape[0],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])

In [46]:
%%time
# Extract all 4 state elements from the 's' column array/list for clean input
X = np.vstack(data['s'].values)
y = data['a'].values

h = model.fit(X, y, epochs=25, verbose=False, validation_split=0.2)

/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)


CPU times: user 2.93 s, sys: 85.1 ms, total: 3.02 s
Wall time: 2.9 s


In [47]:
res = pd.DataFrame(h.history)
res.tail(3)

,acc,loss,val_acc,val_loss
22,0.660688,0.604964,0.690476,0.620654
23,0.662182,0.603614,0.690476,0.619546
24,0.662182,0.602227,0.684524,0.618436


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [49]:
def epoch():
  done = False
  state, _ = env.reset()
  treward = 0
  while not done:
    # Predict expects a batch; np.atleast_2d(state) yields shape (1, 4)
    prob = model.predict(np.atleast_2d(state), verbose=False)[0][0]
    action = 1 if prob > 0.5 else 0

    state, reward, terminated, truncated, _ = env.step(action)
    treward += reward
    done = terminated or truncated
  return treward

In [50]:
res = np.array([epoch() for _ in range(100)])
res

array([ 9., 10., 10., 10., 10., 11., 10.,  9., 10., 10.,  9., 10., 10.,
       10., 11., 10., 11., 11., 10., 10.,  9., 10., 10., 10.,  9., 10.,
       10., 10., 10., 10.,  9.,  9., 10., 10., 10.,  9., 10., 11.,  9.,
       10.,  9.,  9., 10.,  9.,  9., 10.,  9.,  9., 10.,  9.,  9., 10.,
        9.,  9.,  9., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,
       10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.,  9.,  9.,
        9., 10., 10., 10.,  9., 10., 10., 10., 10., 10.,  9., 10.,  9.,
       10., 11., 10., 10., 10., 10., 11., 11., 10.])

In [51]:
res.mean()

np.float64(9.82)